<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 02 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">Customer Geographic Analysis</div>
  <p class="doris-cover-lead">Join customer addresses and orders from two Doris databases, then produce customer, order, and revenue metrics by state.</p>
  <span class="doris-cover-note">Cross-database Source · View · ref() · Table · Data Test</span>
</div>

## 1. Check the execution environment

Run this cell first. It uses the Demo dbt environment and current Doris connection settings, then confirms that a Backend is available.

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("Start Jupyter from the dbt-for-apache-doris repository or a subdirectory.")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 2: Customer Geographic Analysis

Regional operations needs to understand where customers and revenue are concentrated to guide marketing, customer programs, and sales coverage. This Demo uses default shipping addresses and valid orders to create one regional business row per state.

<table class="doris-index">
  <tr><th>Business users</th><td>Regional operations, marketing, and sales management</td></tr>
  <tr><th>Business question</th><td>Which states concentrate customers, valid orders, and revenue, and how do customer value and order frequency compare?</td></tr>
  <tr><th>Metric rule</th><td>Assign customers by default shipping state; keep COMPLETED, DELIVERED, and SHIPPED orders</td></tr>
  <tr><th>Delivered dataset</th><td>State-level metrics table <code>fct_state_customers</code></td></tr>
</table>

The cells below show how two Doris Sources flow through staging Views and are joined with `ref()` into state-level metrics.

<div class="doris-flow">
  <div class="doris-flow-step"><strong>Two Sources</strong>Address table + order table</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Staging Views</strong>Keep default addresses and valid orders</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong><code>ref()</code> join</strong>Join both paths by customer</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>State metrics</strong><code>fct_state_customers</code></div>
</div>

### 2.1 Prepare and inspect the two source tables

The source data lives in two Doris databases. The address table has four rows, including one non-default shipping address; the order table has four rows, including one cancelled order.

In [ ]:
geo_dir = runner.examples_root / "doris-demos/geographic"
runner.show_file("Fixture SQL", geo_dir / "scripts/setup.sql")
runner.run_sql_file("Create geographic source tables", geo_dir / "scripts/setup.sql")
runner.query("Input: customer addresses", """
select address_id, customer_id, state_province, is_default_shipping
from dbt_demo_geographic_customer.CUSTOMER_ADDRESSES
order by address_id
""")
runner.query("Input: orders", """
select order_id, customer_id, grand_total, status
from dbt_demo_geographic_orders.ORDERS
order by order_id
""")

### 2.2 Create staging Views

The two staging models read their source tables with `source()`: the address model keeps default shipping addresses, and the order model keeps COMPLETED, DELIVERED, and SHIPPED orders.

In [ ]:
runner.show_file("Address staging model", geo_dir / "models/stg_customer_addresses.sql")
runner.show_file("Order staging model", geo_dir / "models/stg_orders.sql")
runner.show_file("Source declarations", geo_dir / "models/sources.yml")
runner.run_dbt("Create two staging Views", geo_dir, "run", "--select", "stg_customer_addresses", "stg_orders")
runner.query("Intermediate result: staging Views", """
select 'address' as stage, cast(address_id as string) as record_id, state_province as value
from dbt_demo_geographic.stg_customer_addresses
union all
select 'order', cast(order_id as string), cast(grand_total as string)
from dbt_demo_geographic.stg_orders
order by stage, record_id
""")

### 2.3 Build state metrics with `ref()`

The fact model joins the two staging Views. The result includes state-level scale, average order value, revenue per customer, and orders per customer: CA has two customers, two valid orders, and 145.00 revenue; NY has one customer, one valid order, and 50.00 revenue.

In [ ]:
runner.show_file("State metrics model", geo_dir / "models/fct_state_customers.sql")
runner.run_dbt("Create fct_state_customers", geo_dir, "run", "--select", "fct_state_customers")
runner.query("Output: state customers, orders, revenue, and per-customer metrics", """
select state_province, customer_count, order_count, total_revenue,
       avg_order_value, revenue_per_customer, orders_per_customer
from dbt_demo_geographic.fct_state_customers
order by state_province
""")

### 2.4 Run the Data Test and verify object types

The Data Test checks that state names and customer counts are non-null. The verifier also checks that the final object is a Table and both intermediate objects are Views.

In [ ]:
runner.show_file("Data Test definition", geo_dir / "models/geographic.yml")
runner.run_dbt("Test fct_state_customers", geo_dir, "test", "--select", "fct_state_customers")
runner.run_script("Verify geographic Demo", geo_dir / "scripts/verify.sh")
runner.query("Final object types", """
select table_name, table_type
from information_schema.tables
where table_schema = 'dbt_demo_geographic'
order by table_name
""")

## Complete

The two Sources, two staging Views, state metrics Table, and Data Test all passed verification.